# String Algorithms

Notes and runnable examples on **pattern matching** — finding a pattern inside a text — and the linear-time classics (KMP, Rabin-Karp) that beat the naive O(n·m) scan. The Python reality: `str.find` and `in` are already a tuned C implementation, so you rarely write these — but the ideas (the prefix function, rolling hashes) show up in streaming search, multi-pattern matching, and plenty of interviews.

**Contents**

1. **The problem** — substring search
2. **Naive search**
3. **KMP** — never re-read the text
4. **Rabin-Karp** — rolling hash
5. **What Python actually does**
6. **When to use what**
7. **Complexity cheat-sheet**

## 1. The problem — substring search

The core task: find every position where a **pattern** of length m occurs in a **text** of length n. The obvious approach — try the pattern at each of the n positions — costs **O(n·m)** in the worst case. The classic algorithms cut this to **O(n + m)** by never wasting comparisons: after a partial match fails, they *reuse* what they already learned instead of starting over.

Worth saying up front: in Python you'd just write `pattern in text` or `text.find(pattern)`, which run a tuned **C** implementation (section 5). These algorithms are here for *understanding* — and because their machinery (the prefix function, rolling hashes) powers things the built-ins don't: streaming/online search, many-pattern matching, approximate matching, suffix structures.

## 2. Naive search

Slide the pattern across the text; at each offset, compare characters until they mismatch or the pattern runs out. Simple and correct, but it re-examines text characters it has already seen. Its worst case is a pattern that *almost* matches everywhere — like searching `"aa…aab"` for `"aaab"`:

In [1]:
def naive_search(text, pattern):
    n, m = len(text), len(pattern)
    matches = []
    for i in range(n - m + 1):                 # each possible start position
        j = 0
        while j < m and text[i + j] == pattern[j]:
            j += 1
        if j == m:
            matches.append(i)
    return matches

print("naive_search('abracadabra', 'abra'):", naive_search("abracadabra", "abra"))

# the pathological case: many partial matches -> O(n*m) work
text, pattern = "a" * 20 + "b", "a" * 5 + "b"
print("worst-case input, still correct:", naive_search(text, pattern))


naive_search('abracadabra', 'abra'): [0, 7]
worst-case input, still correct: [15]


## 3. KMP — never re-read the text

**Knuth–Morris–Pratt** makes search **O(n + m)** by precomputing, *from the pattern alone*, how far it can shift after a mismatch without missing anything. That's the **prefix function** (a.k.a. failure function): `pi[i]` is the length of the longest proper prefix of `pattern[:i+1]` that is also a suffix of it. On a mismatch at pattern position `k`, you fall back to `pi[k-1]` instead of restarting — so the **text pointer never moves backward**.

In [2]:
def prefix_function(pattern):
    m = len(pattern)
    pi = [0] * m
    k = 0                                       # length of the current matched prefix
    for i in range(1, m):
        while k > 0 and pattern[i] != pattern[k]:
            k = pi[k - 1]                       # fall back to the next-longest prefix
        if pattern[i] == pattern[k]:
            k += 1
        pi[i] = k
    return pi

def kmp_search(text, pattern):
    if not pattern:
        return [0]
    pi = prefix_function(pattern)
    matches, k = [], 0
    for i, ch in enumerate(text):               # i scans the text and NEVER backtracks
        while k > 0 and ch != pattern[k]:
            k = pi[k - 1]
        if ch == pattern[k]:
            k += 1
        if k == len(pattern):
            matches.append(i - k + 1)
            k = pi[k - 1]                        # keep going, allowing overlaps
    return matches

print("prefix_function('ababaca')   :", prefix_function("ababaca"))
print("kmp_search('abababab','abab'):", kmp_search("abababab", "abab"))   # overlapping hits


prefix_function('ababaca')   : [0, 0, 1, 2, 3, 0, 1]
kmp_search('abababab','abab'): [0, 2, 4]


The prefix function is useful on its own — it exposes a string's internal repetition. For instance, the smallest period of a pattern is `m - pi[m-1]` (when that divides `m`), which is why the same preprocessing turns up far beyond search.

## 4. Rabin-Karp — rolling hash

**Rabin-Karp** takes a different angle: compare a **hash** of the pattern against the hash of each text window. Re-hashing each window from scratch would be O(m), but a **rolling hash** updates it in O(1) — subtract the outgoing character, add the incoming one. When two hashes match, verify the actual characters (to rule out a collision). Average **O(n + m)**; worst case O(n·m) if an adversary forces hash collisions.

Its real strength is **multiple patterns** (hash them all, scan once) and **2D / fingerprinting** problems.

In [3]:
def rabin_karp(text, pattern, base=256, mod=(1 << 61) - 1):
    n, m = len(text), len(pattern)
    if m == 0 or m > n:
        return []
    high = pow(base, m - 1, mod)                # place value of the leading character
    p_hash = t_hash = 0
    for i in range(m):                          # hash the pattern and the first window
        p_hash = (p_hash * base + ord(pattern[i])) % mod
        t_hash = (t_hash * base + ord(text[i])) % mod

    matches = []
    for i in range(n - m + 1):
        if p_hash == t_hash and text[i:i + m] == pattern:    # verify to guard collisions
            matches.append(i)
        if i < n - m:                           # roll the window: drop text[i], add text[i+m]
            t_hash = ((t_hash - ord(text[i]) * high) * base + ord(text[i + m])) % mod
    return matches

print("rabin_karp('abracadabra', 'abra'):", rabin_karp("abracadabra", "abra"))


rabin_karp('abracadabra', 'abra'): [0, 7]


## 5. What Python actually does

You won't beat the built-ins in pure Python. `x in s`, `s.find`, `s.index`, `s.count`, and `s.replace` all call CPython's C **`fastsearch`** (in `stringlib`): a **Boyer-Moore-Horspool**-style skip table, plus (since 3.10) the **two-way** algorithm for long patterns — linear worst-case time with a tiny constant factor. So a pure-Python KMP, despite the same O(n + m) bound, loses badly on the constant:

In [4]:
import timeit

text = "ab" * 500_000 + "c"       # ~1,000,001 characters
pattern = "ab" * 40 + "c"

t_builtin = timeit.timeit(lambda: text.find(pattern), number=20) / 20
t_kmp     = timeit.timeit(lambda: kmp_search(text, pattern), number=20) / 20
print(f"built-in str.find : {t_builtin*1000:7.3f} ms")
print(f"pure-Python KMP   : {t_kmp*1000:7.3f} ms   ({t_kmp/t_builtin:.0f}x slower)")


built-in str.find :   0.960 ms
pure-Python KMP   :  37.654 ms   (39x slower)


The takeaway isn't "KMP is bad" — it's that **the right tool in Python is the built-in**. Hand-write a matcher only when you need something the built-ins can't do: searching a **stream** you can't hold in memory, matching **thousands of patterns** at once (Aho-Corasick generalizes KMP for exactly this), or **approximate/fuzzy** matching (which is edit distance — the DP notebook).

## 6. When to use what

| Situation | Reach for | Notes |
|---|---|---|
| Find a substring in Python | `in` / `.find` / `.index` | C `fastsearch` — just use it |
| Count / replace occurrences | `.count` / `.replace` | same C engine |
| Understanding / interviews | KMP | O(n+m); the prefix function is the core idea |
| Many patterns at once | Aho-Corasick | a KMP automaton built over all patterns |
| Streaming / online search | KMP or rolling hash | process char-by-char, no full text in memory |
| Fingerprinting / 2D / dedup | Rabin-Karp | rolling hashes compose nicely |
| Approximate / fuzzy match | edit distance (DP notebook) | a different problem entirely |

## 7. Complexity cheat-sheet

| Algorithm | Preprocess | Search | Space | Note |
|---|:---:|:---:|:---:|---|
| Naive | — | O(n·m) | O(1) | fine for short text / pattern |
| KMP | O(m) | O(n) | O(m) | text pointer never backtracks |
| Rabin-Karp | O(m) | O(n) avg, O(n·m) worst | O(1) | shines for multi-pattern |
| CPython `fastsearch` | O(m) | O(n) | O(1) | Boyer-Moore-Horspool + two-way, in C |

<sub>n = text length, m = pattern length. The built-in is the one you actually use; the rest are for understanding and for jobs the built-in doesn't cover.</sub>